# ML Pipeline DAG

This notebook creates an automated ML retraining pipeline using Snowflake Tasks. The pipeline implements the **Champion/Challenger pattern** - a production ML best practice where new models must prove they're better than the current production model before being promoted.

## Pipeline Overview

```
┌──────────────────────────┐
│ TASK_TRAIN_CHALLENGER    │  (Root Task - Daily 2AM PST)
│ Train new "challenger"   │
│ model and register it    │
└───────────┬──────────────┘
            ▼
┌──────────────────────────┐
│ TASK_VALIDATE_CHALLENGER │  (Runs after training)
│ Compare challenger AUC   │
│ vs current champion      │
└───────────┬──────────────┘
            ▼
┌──────────────────────────┐
│ TASK_PROMOTE_IF_BETTER   │  (Runs after validation)
│ If challenger > champion │
│ by >1%, promote it       │
└──────────────────────────┘
```

## Prerequisites

This pipeline reuses artifacts created in earlier notebooks:
- `FRAUD_TRAINING_DATA` table (from `02_feature_engineering.ipynb`)
- `FRAUD_DETECTION_MODEL` in Model Registry (from `03a_model_train.ipynb`)

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from datetime import datetime

session = get_active_session()
session.sql("USE ROLE ACCOUNTADMIN").collect()

DATABASE = "DEMO_FRAUD_DETECTION_DB"
SCHEMA = "ML_PIPELINE"
WAREHOUSE = "ML_WH"
COMPUTE_POOL = "ML_TRAINING_POOL"
MODEL_NAME = "FRAUD_DETECTION_MODEL"
SOURCE_STAGE = f"@{DATABASE}.{SCHEMA}.ML_MODELS/source"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
model_ref = reg.get_model(MODEL_NAME)
versions_df = model_ref.show_versions()
latest = versions_df.sort_values('created_on', ascending=False).iloc[0]
LATEST_VERSION = latest['name']

metrics = model_ref.version(LATEST_VERSION).show_metrics()
LATEST_AUC = metrics.get('auc', 0.0) if isinstance(metrics, dict) else 0.0

print(f"Context: {DATABASE}.{SCHEMA}")
print(f"Current best model: {LATEST_VERSION} (AUC={LATEST_AUC})")
print(f"Compute pool: {COMPUTE_POOL}")

## 1. Setup Tracking Tables

Create tables to track pipeline runs and the current champion model:

- **PIPELINE_RUN_LOG**: Records each step of every pipeline run (train, validate, promote)
- **MODEL_CHAMPION_TRACKER**: Tracks the current champion model version and its AUC score

In [ ]:
session.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE}.{SCHEMA}.PIPELINE_RUN_LOG (
    RUN_ID VARCHAR, RUN_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    STEP_NAME VARCHAR, STATUS VARCHAR, CHALLENGER_VERSION VARCHAR,
    CHAMPION_VERSION VARCHAR, CHALLENGER_AUC FLOAT, CHAMPION_AUC FLOAT,
    PROMOTED BOOLEAN, MESSAGE VARCHAR
)""").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE}.{SCHEMA}.MODEL_CHAMPION_TRACKER (
    MODEL_NAME VARCHAR PRIMARY KEY, CHAMPION_VERSION VARCHAR,
    CHAMPION_AUC FLOAT, UPDATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)""").collect()

session.sql(f"""
    MERGE INTO {DATABASE}.{SCHEMA}.MODEL_CHAMPION_TRACKER t
    USING (SELECT '{MODEL_NAME}' AS MODEL_NAME, '{LATEST_VERSION}' AS VER, {LATEST_AUC} AS AUC) s
    ON t.MODEL_NAME = s.MODEL_NAME
    WHEN MATCHED THEN UPDATE SET CHAMPION_VERSION = s.VER, CHAMPION_AUC = s.AUC, UPDATED_AT = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN INSERT VALUES (s.MODEL_NAME, s.VER, s.AUC, CURRENT_TIMESTAMP())
""").collect()

champ = session.sql(f"SELECT * FROM MODEL_CHAMPION_TRACKER WHERE MODEL_NAME = '{MODEL_NAME}'").collect()[0]
print(f"Champion: {champ['CHAMPION_VERSION']} (AUC={champ['CHAMPION_AUC']})")

## 2. Upload Training Code & Create Challenger Procedure

Instead of duplicating training logic, the stored procedure reuses `source/train.py` (the same script used by notebook 03a) by:
1. Uploading `source/` to a stage once
2. The proc downloads it to `/tmp`, then calls `submit_directory()` to run as an **ML Job on SPCS**
3. Passes `VERSION_NAME` and `REGISTER_MODEL=true` as env vars so `train.py` registers the challenger directly
4. Polls until the job completes, then reads metrics from the stage artifacts

This eliminates code duplication — `source/train.py` is the **single source of truth** for training logic.

In [ ]:
import os

session.sql(f"CREATE STAGE IF NOT EXISTS {DATABASE}.{SCHEMA}.ML_MODELS").collect()

source_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'source')
if not os.path.exists(source_dir):
    source_dir = '../source'

for f in os.listdir(source_dir):
    if f.endswith('.py'):
        fpath = os.path.join(source_dir, f)
        session.file.put(fpath, SOURCE_STAGE, auto_compress=False, overwrite=True)
        print(f"Uploaded {f} to {SOURCE_STAGE}")

print(f"\nSource code staged at {SOURCE_STAGE}")

train_sql = f'''
CREATE OR REPLACE PROCEDURE {DATABASE}.{SCHEMA}.TRAIN_CHALLENGER_MODEL()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = \'3.11\'
PACKAGES = (\'snowflake-snowpark-python\', \'snowflake-ml-python\', \'xgboost\', \'scikit-learn\', \'pandas\')
HANDLER = \'run\'
EXECUTE AS CALLER
AS
$$
import os, time, pickle
from datetime import datetime
from snowflake.snowpark import Session

def run(session: Session) -> str:
    DATABASE = "{DATABASE}"
    SCHEMA = "{SCHEMA}"
    COMPUTE_POOL = "{COMPUTE_POOL}"
    MODEL_STAGE = f"@{{DATABASE}}.{{SCHEMA}}.ML_MODELS"
    SOURCE_STAGE = f"@{{DATABASE}}.{{SCHEMA}}.ML_MODELS/source"
    
    version = f"CHALLENGER_{{datetime.now().strftime(\'%Y%m%d_%H%M%S\')}}"
    
    os.makedirs("/tmp/source", exist_ok=True)
    session.file.get(SOURCE_STAGE, "/tmp/source/")
    
    from snowflake.ml.jobs import submit_directory
    job = submit_directory(
        session=session,
        dir_path="/tmp/source",
        entrypoint="train.py",
        compute_pool=COMPUTE_POOL,
        stage_name=f"{{DATABASE}}.{{SCHEMA}}.ML_JOB_STAGE",
        env_vars={{
            "DATABASE": DATABASE,
            "SCHEMA": SCHEMA,
            "MODEL_STAGE": MODEL_STAGE,
            "EXPERIMENT_NAME": "FRAUD_DETECTION_EXPERIMENT",
            "VERSION_NAME": version,
            "REGISTER_MODEL": "true",
        }}
    )
    
    for _ in range(360):
        st = job.status
        if st in ("DONE", "FAILED"):
            break
        time.sleep(10)
    
    if job.status == "FAILED":
        session.sql(f"INSERT INTO {{DATABASE}}.{{SCHEMA}}.PIPELINE_RUN_LOG (RUN_ID, STEP_NAME, STATUS, CHALLENGER_VERSION, MESSAGE) VALUES (CURRENT_TIMESTAMP()::VARCHAR, \'TRAIN\', \'FAILED\', \'{{version}}\', \'ML Job failed\')").collect()
        return f"FAILED: {{version}}"
    
    session.file.get(f"{{MODEL_STAGE}}/metrics.pkl", "/tmp/")
    try:
        with open("/tmp/metrics.pkl", "rb") as f:
            metrics = pickle.load(f)
        auc = metrics.get("auc_roc", 0.0)
        recall = metrics.get("recall", 0.0)
    except:
        auc, recall = 0.0, 0.0
    
    session.sql(f"INSERT INTO {{DATABASE}}.{{SCHEMA}}.PIPELINE_RUN_LOG (RUN_ID, STEP_NAME, STATUS, CHALLENGER_VERSION, CHALLENGER_AUC, MESSAGE) VALUES (CURRENT_TIMESTAMP()::VARCHAR, \'TRAIN\', \'SUCCESS\', \'{{version}}\', {{auc}}, \'AUC={{auc:.4f}}, Recall={{recall:.4f}}\')").collect()
    
    return f"SUCCESS: {{version}} (AUC={{auc:.4f}})"
$$
'''
session.sql(train_sql).collect()
print("Created TRAIN_CHALLENGER_MODEL (uses submit_directory -> source/train.py)")

## 3. Validate & Promote Procedures

### VALIDATE_CHALLENGER
Compares the newly trained challenger model against the current champion:
- Retrieves challenger AUC from the latest TRAIN log entry
- Retrieves champion AUC from `MODEL_CHAMPION_TRACKER`
- Calculates improvement percentage
- Logs decision: `PROMOTE` if improvement >1%, else `KEEP_CHAMPION`

### PROMOTE_IF_BETTER
Conditionally promotes the challenger to become the new champion:
- If validation decided `PROMOTE`: Updates `MODEL_CHAMPION_TRACKER` with new champion
- If validation decided `KEEP_CHAMPION`: Logs skip and retains current champion

In [ ]:
validate_sql = f'''
CREATE OR REPLACE PROCEDURE {DATABASE}.{SCHEMA}.VALIDATE_CHALLENGER()
RETURNS STRING LANGUAGE PYTHON RUNTIME_VERSION = \'3.11\'
PACKAGES = (\'snowflake-snowpark-python\') HANDLER = \'run\'
AS
$$
from snowflake.snowpark import Session
def run(session: Session) -> str:
    DB, SCH = "{DATABASE}", "{SCHEMA}"
    last = session.sql(f"SELECT CHALLENGER_VERSION, CHALLENGER_AUC FROM {{DB}}.{{SCH}}.PIPELINE_RUN_LOG WHERE STEP_NAME=\'TRAIN\' AND STATUS=\'SUCCESS\' ORDER BY RUN_TIMESTAMP DESC LIMIT 1").collect()
    if not last: return "No challenger"
    c_ver, c_auc = last[0]["CHALLENGER_VERSION"], float(last[0]["CHALLENGER_AUC"])
    champ = session.sql(f"SELECT CHAMPION_VERSION, CHAMPION_AUC FROM {{DB}}.{{SCH}}.MODEL_CHAMPION_TRACKER WHERE MODEL_NAME=\'FRAUD_DETECTION_MODEL\'").collect()[0]
    ch_ver, ch_auc = champ["CHAMPION_VERSION"], float(champ["CHAMPION_AUC"])
    improve = (c_auc - ch_auc) / ch_auc
    promote = improve > 0.01
    status = "PROMOTE" if promote else "KEEP_CHAMPION"
    msg = f"Challenger={{c_auc:.4f}} vs Champion={{ch_auc:.4f}} ({{improve*100:+.2f}}%)"
    session.sql(f"INSERT INTO {{DB}}.{{SCH}}.PIPELINE_RUN_LOG (RUN_ID, STEP_NAME, STATUS, CHALLENGER_VERSION, CHAMPION_VERSION, CHALLENGER_AUC, CHAMPION_AUC, PROMOTED, MESSAGE) VALUES (CURRENT_TIMESTAMP()::VARCHAR, \'VALIDATE\', \'{{status}}\', \'{{c_ver}}\', \'{{ch_ver}}\', {{c_auc}}, {{ch_auc}}, {{promote}}, \'{{msg}}\')").collect()
    return f"{{status}}: {{msg}}"
$$
'''
session.sql(validate_sql).collect()
print("Created VALIDATE_CHALLENGER")

In [ ]:
promote_sql = f'''
CREATE OR REPLACE PROCEDURE {DATABASE}.{SCHEMA}.PROMOTE_IF_BETTER()
RETURNS STRING LANGUAGE PYTHON RUNTIME_VERSION = \'3.11\'
PACKAGES = (\'snowflake-snowpark-python\') HANDLER = \'run\'
AS
$$
from snowflake.snowpark import Session
def run(session: Session) -> str:
    DB, SCH = "{DATABASE}", "{SCHEMA}"
    last = session.sql(f"SELECT CHALLENGER_VERSION, CHALLENGER_AUC, PROMOTED FROM {{DB}}.{{SCH}}.PIPELINE_RUN_LOG WHERE STEP_NAME=\'VALIDATE\' ORDER BY RUN_TIMESTAMP DESC LIMIT 1").collect()
    if not last: return "No validation"
    if not last[0]["PROMOTED"]:
        session.sql(f"INSERT INTO {{DB}}.{{SCH}}.PIPELINE_RUN_LOG (RUN_ID, STEP_NAME, STATUS, MESSAGE) VALUES (CURRENT_TIMESTAMP()::VARCHAR, \'PROMOTE\', \'SKIPPED\', \'Champion retained\')").collect()
        return "Champion retained"
    c_ver, c_auc = last[0]["CHALLENGER_VERSION"], float(last[0]["CHALLENGER_AUC"])
    session.sql(f"UPDATE {{DB}}.{{SCH}}.MODEL_CHAMPION_TRACKER SET CHAMPION_VERSION=\'{{c_ver}}\', CHAMPION_AUC={{c_auc}}, UPDATED_AT=CURRENT_TIMESTAMP() WHERE MODEL_NAME=\'FRAUD_DETECTION_MODEL\'").collect()
    session.sql(f"INSERT INTO {{DB}}.{{SCH}}.PIPELINE_RUN_LOG (RUN_ID, STEP_NAME, STATUS, CHALLENGER_VERSION, CHALLENGER_AUC, PROMOTED, MESSAGE) VALUES (CURRENT_TIMESTAMP()::VARCHAR, \'PROMOTE\', \'SUCCESS\', \'{{c_ver}}\', {{c_auc}}, TRUE, \'New champion: {{c_ver}}\')").collect()
    return f"PROMOTED: {{c_ver}}"
$$
'''
session.sql(promote_sql).collect()
print("Created PROMOTE_IF_BETTER")

## 4. Create Task DAG

Creates Snowflake Tasks with dependencies to form a DAG:

| Task | Schedule | Depends On | Action |
|------|----------|------------|--------|
| `TASK_TRAIN_CHALLENGER` | Daily 2AM PST | - (root) | Calls `TRAIN_CHALLENGER_MODEL()` |
| `TASK_VALIDATE_CHALLENGER` | - | TASK_TRAIN_CHALLENGER | Calls `VALIDATE_CHALLENGER()` |
| `TASK_PROMOTE_IF_BETTER` | - | TASK_VALIDATE_CHALLENGER | Calls `PROMOTE_IF_BETTER()` |

Child tasks automatically run after their parent completes successfully.

In [ ]:
try:
    session.sql("ALTER TASK DEMO_FRAUD_DETECTION_DB.ML_PIPELINE.TASK_REFRESH_TRAINING_DATA SUSPEND").collect()
    session.sql("ALTER TASK DEMO_FRAUD_DETECTION_DB.ML_PIPELINE.TASK_PROMOTE_IF_BETTER SUSPEND").collect()
    session.sql("ALTER TASK DEMO_FRAUD_DETECTION_DB.ML_PIPELINE.TASK_VALIDATE_CHALLENGER SUSPEND").collect()
    session.sql("ALTER TASK DEMO_FRAUD_DETECTION_DB.ML_PIPELINE.TASK_TRAIN_CHALLENGER SUSPEND").collect()
    print("Suspended old tasks")
except: pass

session.sql(f"CREATE OR REPLACE TASK {DATABASE}.{SCHEMA}.TASK_TRAIN_CHALLENGER WAREHOUSE={WAREHOUSE} SCHEDULE='USING CRON 0 2 * * * America/Los_Angeles' AS CALL TRAIN_CHALLENGER_MODEL()").collect()
print("Created TASK_TRAIN_CHALLENGER (root - daily 2AM)")

session.sql(f"CREATE OR REPLACE TASK {DATABASE}.{SCHEMA}.TASK_VALIDATE_CHALLENGER WAREHOUSE={WAREHOUSE} AFTER {DATABASE}.{SCHEMA}.TASK_TRAIN_CHALLENGER AS CALL VALIDATE_CHALLENGER()").collect()
print("Created TASK_VALIDATE_CHALLENGER")

session.sql(f"CREATE OR REPLACE TASK {DATABASE}.{SCHEMA}.TASK_PROMOTE_IF_BETTER WAREHOUSE={WAREHOUSE} AFTER {DATABASE}.{SCHEMA}.TASK_VALIDATE_CHALLENGER AS CALL PROMOTE_IF_BETTER()").collect()
print("Created TASK_PROMOTE_IF_BETTER")

## 5. Test Pipeline

Manually execute the pipeline to verify it works:
1. Resume all tasks (required before execution)
2. Execute root task which triggers the full DAG
3. Monitor task completion
4. Review pipeline log to verify champion/challenger logic worked correctly

In [ ]:
session.sql(f"ALTER TASK {DATABASE}.{SCHEMA}.TASK_PROMOTE_IF_BETTER RESUME").collect()
session.sql(f"ALTER TASK {DATABASE}.{SCHEMA}.TASK_VALIDATE_CHALLENGER RESUME").collect()
session.sql(f"ALTER TASK {DATABASE}.{SCHEMA}.TASK_TRAIN_CHALLENGER RESUME").collect()
print("Tasks resumed")

In [ ]:
import time
print("Executing pipeline (ML Job takes 5-10 min)...")
session.sql(f"EXECUTE TASK {DATABASE}.{SCHEMA}.TASK_TRAIN_CHALLENGER").collect()

completed = set()
for i in range(180):
    hist = session.sql(f"SELECT NAME, STATE, DATEDIFF('second', SCHEDULED_TIME, COALESCE(COMPLETED_TIME, CURRENT_TIMESTAMP())) as SEC FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(SCHEDULED_TIME_RANGE_START=>DATEADD('minute',-15,CURRENT_TIMESTAMP()))) WHERE DATABASE_NAME='{DATABASE}' AND SCHEMA_NAME='{SCHEMA}' ORDER BY SCHEDULED_TIME DESC").collect()
    for r in hist:
        if r['NAME'] not in completed:
            if r['STATE'] == 'SUCCEEDED':
                print(f"  ✓ {r['NAME']}: {r['SEC']}s")
                completed.add(r['NAME'])
            elif r['STATE'] == 'FAILED':
                print(f"  ✗ {r['NAME']}: FAILED")
                completed.add(r['NAME'])
    if len(completed) >= 3: break
    time.sleep(5)

print("\n📋 Pipeline Log:")
log = session.sql(f"SELECT TO_CHAR(RUN_TIMESTAMP,'HH24:MI:SS') as T, STEP_NAME, STATUS, CHALLENGER_VERSION, ROUND(CHALLENGER_AUC,4) as AUC, SUBSTR(MESSAGE,1,50) as MSG FROM PIPELINE_RUN_LOG WHERE RUN_TIMESTAMP > DATEADD('minute',-15,CURRENT_TIMESTAMP()) ORDER BY RUN_TIMESTAMP DESC").to_pandas()
print(log.to_string(index=False))

In [ ]:
print("\n" + "="*60)
print("✅ ML PIPELINE DAG SETUP COMPLETE")
print("="*60)
print("\nPipeline Summary:")
print("  • 3 tasks created with proper dependencies")
print("  • Runs daily at 2AM PST (or manually via EXECUTE TASK)")
print("  • Champion/Challenger validation with 1% improvement threshold")
print("\nTo monitor in Snowsight:")
print("  Data > Databases > DEMO_FRAUD_DETECTION_DB > ML_PIPELINE > Tasks")
print("\nTo suspend pipeline:")
print("  ALTER TASK DEMO_FRAUD_DETECTION_DB.ML_PIPELINE.TASK_TRAIN_CHALLENGER SUSPEND;")